# XAI benchmark v2 su Colab

1. **Runtime solo CPU** (gratuito): passi 1 e 2, i test su modelli finti. Se falliscono, ci si ferma qui.
2. **Runtime GPU**: passo 1, passo 3 (stima dei tempi), poi passo 4, che fa tutto in parallelo.

I risultati vanno su Google Drive. Se Colab si disconnette, rilanciando la stessa cella il lavoro riprende da dove era arrivato, e il risultato è identico a quello di un run senza interruzioni.

## 1. Codice e dipendenze

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q --branch v2 https://github.com/nicolobachiorri/XAI_Benchmark_for_Sentiment_Analysis.git /content/xai
%cd /content/xai
!git log --oneline -1

Mounted at /content/drive
/content/xai
47dbaf1 (HEAD -> v2, origin/v2) Test compatibili con transformers 5; contrastivity anche come Jensen-Shannon


In [2]:
!pip install -q lime pytest
import os, torch, transformers
print(torch.__version__, transformers.__version__, 'GPU' if torch.cuda.is_available() else 'CPU', os.cpu_count(), 'CPU')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 22.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
2.11.0+cu130 5.18.0 GPU 2 CPU


## 2. Test (CPU, pochi minuti, niente download)

In [13]:
!python -m pytest -q tests

.............................................................            [100%]
=============================== warnings summary ===============================
tests/test_torch.py::test_grad_input_matches_manual_computation
  /content/xai/tests/test_torch.py:205: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
  Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
    assert np.isclose(sum(a.raw_scores), float(per_token[1:-1].sum()), rtol=1e-4, atol=1e-6)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
61 passed, 1 warning in 167.67s (0:02:47)


## 3. Stima dei tempi sull'hardware in uso (GPU)

Scarica i cinque modelli e cronometra ogni metodo su tre recensioni di lunghezza tipica. Il piano (passo 4 con `--dry-run`) mostra come verrà diviso il lavoro tra GPU e CPU.

In [3]:
!python report.py --estimate
!python parallel_run.py --out-dir /content/drive/MyDrive/xai_v2 --dry-run

[DATA] 400 testi (188 positivi), device cuda
[ESTIMATE] tinybert: caricamento...
config.json: 100% 844/844 [00:00<00:00, 3.37MB/s]

model.safetensors: downloading bytes:  89% 51.2M/57.4M [00:06<00:00, 18.2MB/s, 3.73MB/s  ]
model.safetensors: downloading bytes: 100% 54.8M/54.8M [00:07<00:00, 7.78MB/s, 4.25MB/s  ]
model.safetensors: reconstructing file: 100% 57.4M/57.4M [00:07<00:00, 8.16MB/s, 4.90MB/s  ]
Loading weights: 100% 73/73 [00:00<00:00, 6071.35it/s]
tokenizer_config.json: 100% 1.30k/1.30k [00:00<00:00, 4.38MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 64.0MB/s]
tokenizer.json: 100% 712k/712k [00:00<00:00, 41.2MB/s]
special_tokens_map.json: 100% 125/125 [00:00<00:00, 618kB/s]
  lime                 0.917 s/spiegazione  ->  1h 7m per 400 testi
  shap                 0.058 s/spiegazione  ->  4m 16s per 400 testi
  grad_input           0.013 s/spiegazione  ->  55.6s per 400 testi
  attention_rollout    0.016 s/spiegazione  ->  1m 11s per 400 testi
  attention_flow       0.809 s/spi

## 4. Run completo, tutto in parallelo

Sulla GPU girano insieme due processi con LIME, SHAP, Input×Gradient, Rollout e LRP; sulle CPU libere girano i processi di Attention Flow. Una riga di avanzamento ogni 5 minuti; i log dei singoli processi sono in `xai_v2/logs/`. Alla fine stampa le tre tabelle.

In [ ]:
!python parallel_run.py --out-dir /content/drive/MyDrive/xai_v2

### Variante: due runtime insieme

Se il runtime GPU ha poche CPU, Attention Flow diventa la parte più lunga. Si può aprire una seconda copia del notebook su un runtime solo CPU e dividere il lavoro sulla stessa cartella di Drive:

- runtime GPU: `!python parallel_run.py --role gpu --out-dir /content/drive/MyDrive/xai_v2`
- runtime CPU: `!python parallel_run.py --role flow --out-dir /content/drive/MyDrive/xai_v2`

Quando hanno finito tutti e due, la cella seguente ricostruisce le tabelle complete.

In [ ]:
!python report.py --summarize-only --out-dir /content/drive/MyDrive/xai_v2